In this script, we analyze the impact of batch size on inference speed. The objective is to quantify the performance gain obtained by processing multiple images simultaneously rather than sequentially.

We will specifically observe:
* The overall acceleration of the inference pipeline thanks to batching.
* The trade-off between model complexity (number of parameters) and the resulting accuracy.

In [1]:
# Only import fovea as it provides all necessary utilities
import retinoto_py as fovea
subset_factor = 2
args = fovea.Params(do_fovea=True, subset_factor=subset_factor)
args

Params(batch_size=128, num_workers=0, prefetch_factor=0, image_size=224, grid_size_ecc=161, grid_size_ang=309, do_mask=False, do_fovea=False, use_hexagonal_grid=True, rs_min=-3.5, rs_max=0.7, angle_start=-5.235987755982989, angle_margin=0.010166966516471823, mode='bilinear', padding_mode='zeros', model_name='convnext_base', num_epochs=340, subset_factor=1, optimizer_name='adamw', loss_name='CrossEntropyLoss', base_lr=1e-06, final_lr=1e-09, num_warmup_epochs=20, delta1=0.3, delta2=0.002, weight_decay=0.0001, label_smoothing=0.0005, do_full_training=True, do_augment=True, augment_proba=0.85, stochastic_depth_prob=0.6, seed=1998, shuffle=True, verbose=False)

# testing the ConvNet model on the validation dataset

In [2]:
VAL_DATA_DIR = args.DATAROOT / 'Imagenet_full' / 'val'

In [3]:
# Load model and benchmark its complexity (parameters and layers)
model = fovea.load_model(args)
model.eval()
# --- Parameter Counts ---
param_stats = fovea.count_parameters(model)
print("🔢 Parameter Count:")
print(f"  Total:     {param_stats['total_parameters']:,}")
print(f"  Trainable: {param_stats['trainable_parameters']:,}")
print("-" * 50)

# --- Layer Counts ---
print("🧱 Layer Count:")
total_layers = fovea.count_layers(model)
print(f"  Total Modules (nn.Module): {total_layers}")

from torch import nn
conv_layers = fovea.count_layers(model, layer_type=nn.Conv2d)
linear_layers = fovea.count_layers(model, layer_type=nn.Linear)
print(f"  Convolutional (nn.Conv2d): {conv_layers}")
print(f"  Linear (nn.Linear): {linear_layers}")
print("=" * 50)

🔢 Parameter Count:
  Total:     88,591,464
  Trainable: 88,591,464
--------------------------------------------------
🧱 Layer Count:
  Total Modules (nn.Module): 383
  Convolutional (nn.Conv2d): 40
  Linear (nn.Linear): 73


In [4]:
# 4. Calculate baseline accuracy on the full dataset to ensure model consistency
json_filename = args.data_cache / '11_model_accuracy.json'

if json_filename.exists():
    results = fovea.pd.read_json(json_filename)
else:
    all_results_list = []
    for dataset in ['full']:# <HACK until bbox is finished> fovea.params.all_datasets:
        VAL_DATA_DIR = args.DATAROOT / f'Imagenet_{dataset}' / 'val'
        val_dataset = fovea.get_dataset(args, VAL_DATA_DIR)
        val_loader = fovea.get_loader(args, val_dataset)
        accuracy = fovea.get_validation_accuracy(args, model, val_loader, f"Evaluating {args.model_name} on dataset: {dataset}")
        all_results_list.append({'model_name':args.model_name, 'dataset':dataset, 'accuracy': accuracy})

    results = fovea.pd.DataFrame(all_results_list)
    results['accuracy_str'] = results['accuracy'].apply(lambda x: f"Accuracy: {x * 100:.1f}%")
    results.to_json(json_filename, orient='records', indent=2)

print(f"Evaluation complete.")

Evaluating convnext_base on dataset: full:   0%|          | 0/390 [00:00<?, ?it/s]

Evaluation complete.


In [5]:
results

,model_name,dataset,accuracy,accuracy_str
0,convnext_base,full,0.745713,Accuracy: 74.6%


In [6]:
for _, row in results.iterrows():
    accuracy_percent = row['accuracy'] * 100
    print(f"Accuracy for {row['dataset']}: {accuracy_percent:.3f}%")

Accuracy for full: 74.571%


In [7]:
%cat {json_filename}

[
  {
    "model_name":"convnext_base",
    "dataset":"full",
    "accuracy":0.745713141,
    "accuracy_str":"Accuracy: 74.6%"
  }
]

In [8]:
print(results[['dataset', 'accuracy_str']].to_string(index=False))

dataset    accuracy_str
   full Accuracy: 74.6%
